# 03 · Entrenamiento de la CNN base

Este experimento entrena únicamente la CNN personalizada de Keras. Usa las particiones reproducibles del notebook 02 y solo consulta `train` y `validation` durante el ajuste y la selección del mejor checkpoint. La partición `test` queda reservada para la evaluación final.

Los valores de `configs/baseline.json` son un punto de partida, no hiperparámetros definitivos. El entrenamiento puede ser lento en CPU. La celda de ajuste está desactivada por defecto; actívala explícitamente cuando estés lista para iniciar el entrenamiento.

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

_here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [_here, *_here.parents] if (p / "src").is_dir() and (p / "configs").is_dir()), _here)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import create_stratified_splits, build_tf_dataset
from src.model import build_cnn, compile_classifier
from src.train import set_random_seeds, train_model

CONFIG_PATH = PROJECT_ROOT / "configs" / "baseline.json"
with CONFIG_PATH.open(encoding="utf-8") as file:
    config = json.load(file)

CLASS_ROOT = PROJECT_ROOT / "data" / "raw" / "asl_alphabet_train" / "asl_alphabet_train"
set_random_seeds(config["seed"])
print("Configuración:")
print(json.dumps(config, ensure_ascii=False, indent=2))
print("\nCarpeta de clases:", CLASS_ROOT)
if not CLASS_ROOT.is_dir():
    raise FileNotFoundError(f"No se encontró {CLASS_ROOT}. Ajusta CLASS_ROOT según el notebook 02.")

Configuración:
{
  "seed": 42,
  "validation_fraction": 0.15,
  "test_fraction": 0.15,
  "image_size": [
    200,
    200
  ],
  "batch_size": 32,
  "epochs": 20,
  "early_stopping_patience": 5,
  "learning_rate": 0.001,
  "dropout_rate": 0.4,
  "checkpoint_path": "models/baseline_best.keras",
  "history_path": "results/metrics/baseline_history.csv"
}

Carpeta de clases: C:\SOFIA\ESTUDIOS\UNIVERSIDADES\UIS\PROYECTOS GITHUB\asl-sign-language-recognition\data\raw\asl_alphabet_train\asl_alphabet_train


## 1. Preparar las particiones y los lotes

Se recrean las particiones con la misma semilla y configuración del notebook 02. Esta celda vuelve a calcular hashes para descartar duplicados exactos y puede tardar. No se carga el conjunto de prueba en la llamada de entrenamiento.

In [2]:
splits = create_stratified_splits(
    CLASS_ROOT,
    validation_fraction=config["validation_fraction"],
    test_fraction=config["test_fraction"],
    seed=config["seed"],
)
print(f"Tamaños: train={len(splits.train):,}, validation={len(splits.validation):,}, test reservado={len(splits.test):,}")
print("Clases:", len(splits.class_names), "| duplicados internos removidos:", splits.duplicates_removed)

image_size = tuple(config["image_size"])
train_ds = build_tf_dataset(
    splits.train, image_size=image_size, batch_size=config["batch_size"],
    shuffle=True, seed=config["seed"],
)
validation_ds = build_tf_dataset(
    splits.validation, image_size=image_size, batch_size=config["batch_size"], shuffle=False,
)
print("Lotes train:", len(splits.train) // config["batch_size"] + int(len(splits.train) % config["batch_size"] != 0))
print("Lotes validation:", len(splits.validation) // config["batch_size"] + int(len(splits.validation) % config["batch_size"] != 0))

Tamaños: train=60,900, validation=13,050, test reservado=13,050
Clases: 29 | duplicados internos removidos: 0
Lotes train: 1904
Lotes validation: 408


## 2. Construir el modelo

La última capa produce logits para las 29 clases y `compile_classifier` usa etiquetas enteras (`label_id`) con `SparseCategoricalCrossentropy(from_logits=True`).

In [3]:
model = build_cnn(
    input_shape=(*image_size, 3),
    num_classes=len(splits.class_names),
    dropout_rate=config["dropout_rate"],
)
compile_classifier(model, learning_rate=config["learning_rate"])
model.summary()
print(f"Parámetros: {model.count_params():,}")

Model: "asl_cnn_baseline"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 200, 200, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 200, 200, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_bn1 (BatchNormalization) │ (None, 200, 200, 32)   │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_relu1 (Activation)       │ (None, 200, 200, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 200, 200, 32)   │         9,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_bn2 (BatchNormalization) │ (None, 200, 200, 32)   │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_relu2 (Activation)       │ (None, 200, 200, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 100, 100, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 100, 100, 64)   │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_bn1 (BatchNormalization) │ (None, 100, 100, 64)   │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_relu1 (Activation)       │ (None, 100, 100, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 100, 100, 64)   │        36,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_bn2 (BatchNormalization) │ (None, 100, 100, 64)   │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_relu2 (Activation)       │ (None, 100, 100, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 50, 50, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 50, 50, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_bn1 (BatchNormalization) │ (None, 50, 50, 128)    │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_relu1 (Activation)       │ (None, 50, 50, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 50, 50, 128)    │       147,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_bn2 (BatchNormalization) │ (None, 50, 50, 128)    │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_relu2 (Activation)       │ (None, 50, 50, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 25, 25, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 25, 25, 256)    │       294,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_bn1 (BatchNormalization) │ (None, 25, 25, 256)    │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_relu1 (Activation)       │ (None, 25, 25, 256)    │             

 Total params: 1,211,773 (4.62 MB)

 Trainable params: 1,209,853 (4.62 MB)

 Non-trainable params: 1,920 (7.50 KB)

Parámetros: 1,211,773


## 3. Entrenar solo cuando estés lista

Para iniciar el ajuste, cambia `RUN_TRAINING` a `True` en la celda siguiente. Se guardará el mejor modelo según `val_loss` y un CSV con las métricas de cada época. Early Stopping restaura los mejores pesos. No se pasa `test_ds` a `fit` ni a `train_model`.

In [ ]:
RUN_TRAINING = True

if RUN_TRAINING:
    history = train_model(
        model,
        train_ds,
        validation_ds,
        epochs=config["epochs"],
        patience=config["early_stopping_patience"],
        checkpoint_path=PROJECT_ROOT / config["checkpoint_path"],
        history_path=PROJECT_ROOT / config["history_path"],
    )
    history_frame = pd.DataFrame(history.history)
    display(history_frame.tail())
else:
    print("Entrenamiento no iniciado. Cambia RUN_TRAINING a True cuando quieras ejecutarlo.")

Epoch 1/20


c:\SOFIA\ESTUDIOS\UNIVERSIDADES\UIS\PROYECTOS GITHUB\asl-sign-language-recognition\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


  57/1904 ━━━━━━━━━━━━━━━━━━━━ 2:13:15 4s/step - accuracy: 0.0828 - loss: 3.2584

In [ ]:
if "history" in globals():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history["loss"], label="train")
    axes[0].plot(history.history["val_loss"], label="validation")
    axes[0].set(title="Pérdida", xlabel="Época", ylabel="Loss")
    axes[0].legend()
    axes[1].plot(history.history["accuracy"], label="train")
    axes[1].plot(history.history["val_accuracy"], label="validation")
    axes[1].set(title="Accuracy", xlabel="Época", ylabel="Accuracy")
    axes[1].legend()
    plt.tight_layout()
    plt.show()
else:
    print("Las curvas aparecerán después de entrenar el modelo.")

## Registro del experimento

Después del entrenamiento, documenta el tiempo de ejecución, si terminó por Early Stopping y cualquier problema de memoria o rendimiento. La evaluación final del conjunto `test` se implementará por separado en la siguiente etapa; no uses esos datos para escoger épocas ni ajustar hiperparámetros.